# Asymmetric lungs

Batches that sweep `asymmetric_lung_graph` over `flow_ratio`, at one target size or several, with random matches. Other batches go in `../analysis.ipynb`, whose cross-size section needs one graph per type and size.
Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.

Mean times are conditional on fixation. `fixation_prob` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from moran_graphs.plots import COLORS, LABELS, heatmap, lines, load, print_batch, print_run_report, sweep
from moran_graphs.PopulationGraph import PopulationGraph

The trees at each swept `flow_ratio`, `n_nodes` = 31: trachea on top, one row per generation, major daughter to the left.

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(15, 2.5), layout="constrained")
for ax, flow_ratio in zip(axes, [0.1, 0.15, 0.2, 0.25, 0.3, 0.326, 0.35, 0.4, 0.45, 0.5]):
    lung = PopulationGraph.asymmetric_lung_graph(flow_ratio, n_nodes=31).graph
    nx.draw(lung, nx.get_node_attributes(lung, "pos"), ax=ax, node_size=8, width=0.5,
            node_color=COLORS["Asymmetric lung"])
    ax.set_title(f"{flow_ratio} (N = {lung.number_of_nodes()})", fontsize=8)

## Configuration

The only cell to edit.

In [ ]:
BATCHES = ["2026-10-07-asymmetric-lungs-size-sweep"]
R_VALUES = [1.1]  # the per-r figures need exactly one

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

In [ ]:
results, hists = load(BATCHES)
print(f"{results['hash'].nunique()} graphs, r values {sorted(results['r'].unique().tolist())}")
results = results[results["r"].isin(R_VALUES)]
results.groupby(["category", "r"])[["fixation_prob", "mean_fixation_time", "n_censored"]].agg(["count", "mean", "min", "max"])

## Monte Carlo precision

How well each (graph, r) cell is measured: runs, fixed runs, SE and the relative 95% CI half-width of both metrics, against N. The dashed line is the target half-width `stage_two_plan` sizes n for. Fixed runs set the precision of both metrics, so cells with a small fixation probability are the noisiest.

In [ ]:
from moran_graphs.plots import precision

fig = precision(results)

Fixation-time distributions, one row per `flow_ratio` and a split violin per N: the lung on the left, its matched random cloud on the right. Narrow, untailed distributions need few fixed runs for a precise mean; the numbers at the bottom are the median CV.

In [ ]:
from moran_graphs.plots import time_violins

# one row per flow_ratio; random graphs have none, so they only appear as clouds
flow_ratio = results["kwargs"].map(lambda kwargs: json.loads(kwargs).get("flow_ratio"))
for r in sorted(results["r"].unique()):
    fig = time_violins(results.assign(flow_ratio=flow_ratio), hists, r, rows="flow_ratio")


How much of each random cloud's spread is simulation noise (Monte Carlo variance over the observed variance between its graphs). Near 0 the middle-95% band measures structure; near 1 the band, and the stands-out test built on it, is mostly noise.

In [ ]:
from moran_graphs.plots import random_noise_share

fig = random_noise_share(results)

Stage-two batches only: each cell's estimate against its independent pilot estimate as a z-score. About 5% of cells should fall beyond 1.96; many more means the engine, the seeding or the SE is wrong. Draws nothing for batches without a pilot.

In [ ]:
from moran_graphs.plots import pilot_agreement

fig = pilot_agreement(results)

## Parameter sweep

One category against a kwarg of its recipe, e.g. the asymmetric lungs against `flow_ratio`. Points are colored by N, because `n_nodes` is only a target and the built tree's N moves with the sweep.
Grey bars: the random graphs with the same N and E (middle 95%), at the same x.
Bottom row: each graph over the mean of those random graphs (`*_over_random` columns, added by `load`), which takes N out of the comparison. No error bars there: the spread of the random graphs (grey) is the reference.

Several kwarg values can build the same graph; the duplicates are dropped, so each point is the first value that built its graph (see the `graphs` warning in the verification report).

Dashed line: the human lung, `flow_ratio` = 0.326. In 2026-10-06-asymmetric-lungs it builds the same tree as 0.32, so that point (N = 33) is the human tree.

In [ ]:
SWEEP_CATEGORY, SWEEP_KWARG = "Asymmetric lung", "flow_ratio"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023, the asymmetric_lung_graph default

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), layout="constrained", sharex=True)
ys = ("fixation_prob", "mean_fixation_time", "fixation_prob_over_random", "mean_fixation_time_over_random")
for ax, y in zip(axes.flat, ys):
    points = sweep(ax, results, SWEEP_CATEGORY, SWEEP_KWARG, y)
    if y.endswith("_over_random"):
        ax.axhline(1, color=COLORS["Random"], lw=1)
    ax.axvline(HUMAN_FLOW_RATIO, color="0.2", ls="--", lw=1, label=f"human lung ({HUMAN_FLOW_RATIO})")
fig.colorbar(points, ax=axes, label="N")
axes[0, 0].legend(frameon=False)
fig.suptitle(f"r = {R_VALUES}");

## Asymmetric lungs across sizes

For a batch that sweeps `flow_ratio` at several target sizes (`n_nodes` in the recipe), e.g. 2026-10-07-asymmetric-lungs-size-sweep. One line per `flow_ratio` against the built N, which misses the target by a few percent; grey: the random trees.

- **Left:** fixation probability and mean fixation time (given fixation).
- **Right:** the same divided by the mean random tree with the same N and E (`*_over_random` from `load`), as in `versus_random`. Above 1 the lung fixes more often (top) or slower (bottom) than random. Grey here is each random tree over that same mean: the spread a lung has to leave to stand out.

Error bars are 95% intervals (Wilson for `fixation_prob`, 1.96 SE for time), left only: on the right the reference is the random spread.

In [ ]:
LUNG_CATEGORY = "Asymmetric lung"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023

lungs = results[results["category"] == LUNG_CATEGORY]
recipe = pd.json_normalize(lungs["kwargs"].map(json.loads).tolist()).set_index(lungs.index)
lungs = lungs.assign(flow_ratio=recipe["flow_ratio"], **{"target N": recipe["n_nodes"]})
random = results[results["category"] == "Random"]
human = lungs[lungs["flow_ratio"] == HUMAN_FLOW_RATIO].sort_values("n_nodes")

panels = [(y, LABELS[y], "log" if "time" in y else "linear")
          for y in ("fixation_prob", "fixation_prob_over_random", "mean_fixation_time", "mean_fixation_time_over_random")]
fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout="constrained")
for ax, (y, ylabel, yscale) in zip(axes.flat, panels):
    ax.scatter(random["n_nodes"], random[y], s=8, alpha=0.5, linewidths=0, color=COLORS["Random"],
               label="random trees")
    lines(ax, lungs, "n_nodes", y, by="flow_ratio", cmap="plasma")
    ax.plot(human["n_nodes"], human[y], color="0.1", ls="--", lw=1.5, zorder=4, label="human lung")
    if y.endswith("_over_random"):
        ax.axhline(1, color="0.2", lw=1)
    ax.set(xscale="log", yscale=yscale, ylabel=ylabel)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="outside right center", frameon=False, fontsize=9)
fig.suptitle(f"Asymmetric lungs across sizes against random trees of the same N and E, r = {R_VALUES}");

### Heatmaps over target N and flow_ratio

The four panels of the line plots above, one cell per (target N, `flow_ratio`).
- **Left:** fixation probability and mean fixation time (given fixation). Time is on a log color scale because it grows by orders of magnitude with N, so these panels mostly show N.
- **Right:** the same over the mean random tree with the same N and E, which takes N out: what is left is the effect of `flow_ratio` at each size. Above 1 (red) the lung fixes more often (top) or slower (bottom) than random. Bold: outside the middle 95% of the random trees. Symmetric around 1: linear for fixation probability, log for time (so x2 and x1/2 are equally strong).

Grey: a `flow_ratio` that built the same tree as a smaller one at that size, dropped as a duplicate. Boxed column: the human lung.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 9), layout="constrained")
for ax, (y, label, yscale) in zip(axes.flat, panels):
    grid = heatmap(ax, lungs, "flow_ratio", y, "target N", label, yscale)
    human_column = grid.columns.get_loc(HUMAN_FLOW_RATIO)
    ax.add_patch(plt.Rectangle((human_column - 0.5, -0.5), 1, len(grid), fill=False, lw=1.5))
    ax.tick_params(axis="x", labelrotation=45)
fig.suptitle(f"Asymmetric lungs over target N and flow_ratio, r = {R_VALUES}");